# 💻 Data Visualization — Coding Challenges

**Type:** INTERVIEW PRACTICE · **Level:** 🟠 Intermediate

## About these challenges

The best way to know you understand a data-visualization library is to rebuild its small essentials blind — no hints, no autocomplete. These 10 challenges do exactly that: each one describes a tiny, realistic plotting task (title a line chart, overlay two histograms, sort bars, prepare a correlation matrix, downsample a giant series, label bars, build a facet grid, apply a log scale, save a figure, choose the right chart). 

The important twist:**every solution cell ends with an assertion** (`assert ...`). That means the notebook self-checks — you literally cannot proceed with a wrong answer. The assertion pins the *one* behavior that matters (the title's text, the legend's presence, the scale name, the file size), which is also the pattern used in real engineering tests.

Two strong pieces of advice before you start:

1. **Run each cell alone.** The solutions are small and independent; work top to bottom and let the `assert` be your judge.
2. **Time yourself.** As cell 23 reminds you, 10 minutes per challenge is a realistic screening pace — redo any challenge that took longer.

The challenges deliberately mix plotting *construction* (API calls) with plotting *judgment* (choosing the chart). The final challenge, number 10, is the whole point of the exercise in miniature.

## Setup

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 30)
np.random.seed(0)
print('pandas', pd.__version__)

## Challenge 1

Create a labelled line chart and assert the title.

## Challenge 1 — Dictionary labels on a line chart

**The skill.** Creating a figure-and-axes object and customizing it explicitly. `fig, ax = plt.subplots()` gives you the canvas (`fig`) and the axes (`ax`); `ax.plot(xs, ys)` draws the line; `ax.set_title('Trend')` labels it. The challenge then verifies with `assert ax.get_title() == 'Trend'` — a reminder that titles, like everything, are stored back on the axes object where you can read them programmatically.

**Think through.** Why use `ax.set_title` instead of `plt.title`? Because `ax.set_*` applies to one specific axes — essential when a figure contains many subplots.

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
ax.plot([1, 2, 3], [4, 5, 6])
ax.set_title('Trend')
assert ax.get_title() == 'Trend'
print('ok')

## Challenge 2

Plot two distributions on shared axes with a legend.

## Challenge 2 — Two distributions, shared axes, legend

**The skill.** Overlaying two datasets on the same axes. Drawing a histogram emphasizes the shape via bins; calling `ax.hist(...)` twice with `alpha=0.5` makes the second histogram translucent enough that the first stays visible underneath. Each call gets a `label` and `ax.legend()` draws the key. The assertion `ax.get_legend() is not None` checks the legend actually exists — the classic way an unattended legend goes missing.

**Think through.** The two samples are `rng.normal(0, 1, 200)` and `rng.normal(1, 1, 200)` — the same spread, different means. `alpha` (transparency) is the trick that lets two overlapping histograms be readable at all.

In [ ]:
import numpy as np, matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
fig, ax = plt.subplots()
ax.hist(rng.normal(0, 1, 200), alpha=0.5, label='a')
ax.hist(rng.normal(1, 1, 200), alpha=0.5, label='b')
ax.legend()
assert ax.get_legend() is not None
print('ok')

## Challenge 3

Sort bars by value before plotting.

## Challenge 3 — Sort before you plot

**The skill.** Bars look misleading if you plot them in alphabetical or insertion order, so real honos sort categories by value first. Here no chart is even drawn: the challenge is the *data prep* — turning `{'a': 3, 'b': 9, 'c': 5}` into a dict ordered by descending value. `sorted(d.items(), key=lambda kv: -kv[1])` sorts the (key, value) pairs by negative value, so the biggest comes first, and `dict(...)` keeps that order (Python dicts are insertion-ordered). `assert list(ordered)[0] == 'b'` proves 'b' (the largest) is first.

**Think through.** Sorting is a prerequisite habit for every bar plot: `ax.bar(x_sorted, y_sorted)` with descending order turns a scramble into a clean Pareto curve. Note the trick uses `-kv[1]` instead of `reverse=True` — either works.

In [ ]:
d = {'a': 3, 'b': 9, 'c': 5}
ordered = dict(sorted(d.items(), key=lambda kv: -kv[1]))
assert list(ordered)[0] == 'b'
print(ordered)

## Challenge 4

Build a correlation matrix ready for a heatmap.

## Challenge 4 — A correlation matrix, heatmap-ready

**The skill.** A heatmap eats a *matrix*, not a DataFrame. `df.corr()` computes the Pearson correlation between every pair of numeric columns and returns a square matrix indexed and coloured by column names. For a 3-column frame you get a 3×3 matrix: ones on the diagonal (each column is perfectly correlated with itself) and values in [−1, 1] elsewhere. The assertion checks both facts — `c.shape == (3, 3)` and `np.allclose(np.diag(c), 1)` (diagonal ≈ 1).

**Think through.** After this, the matrix is one line away from `sns.heatmap(...)` or `px.imshow(...)`. Correlations are the standard pre-model check for redundant features.

In [ ]:
import pandas as pd, numpy as np
df = pd.DataFrame(np.random.default_rng(0).normal(size=(50, 3)),
                  columns=list('abc'))
c = df.corr()
assert c.shape == (3, 3) and np.allclose(np.diag(c), 1)
print(c.round(2))

## Challenge 5

Downsample before plotting a large series.

## Challenge 5 — Downsample before you plot

**The skill.** Plotting a 500,000-point series is slow and, worse, pointless — the eye cannot parse half a million glyphs. The professional move is to *sample* first. `s.iloc[::100]` takes every 100th element with a pure-string slice, cutting 500,000 points to 5,000 — `assert len(small) == 5000` proves it. Downsampling keeps the shape of the series while making every rendering instant.

**Think through.** `iloc[::100]` is a step slice, exactly like `[::2]` for every other row; the ratio step = len/original is chosen so the sampled curve still shows the trend. The same idea applies to pyplot data ingestion for time series before any line plot.

In [ ]:
import pandas as pd
s = pd.Series(range(500_000))
small = s.iloc[::100]
assert len(small) == 5000
print(len(small))

## Challenge 6

Add value labels to a bar chart.

## Challenge 6 — Value labels on bars

**The skill.** Numbers on top of bars make a chart self-explanatory. `ax.bar(['x', 'y'], [4, 7])` returns the bar containers; `ax.bar_label(bars)` walks every bar and draws its value just above it — Matplotlib handles position, alignment and formatting automatically. `assert len(labels) == 2` verifies one label per bar.

**Think through.** `bar_label` returns the text objects it created. This is the Matplotlib answer to Plotly's `text_auto` or Seaborn's `annot` from earlier sessions; when your audience must read exact values, label the bars.

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
bars = ax.bar(['x', 'y'], [4, 7])
labels = ax.bar_label(bars)
assert len(labels) == 2
print('ok')

## Challenge 7

Create a faceted grid of four panels.

## Challenge 7 — A faceted grid of panels

**The skill.** Multi-panel figures are the standard cure for overcrowded charts. `plt.subplots(2, 2)` returns a figure plus a 2×2 array of axes; `axes.size` reports the total number of panels, which the assertion checks equals 4. Facets let each category get its own axes (the seaborn `col=`/`row=` trick, here done manually), and indexing `axes[r, c]` lets you reach any single panel.

**Think through.** This is the manual version of what `catplot(col=...)`, `PairGrid` and `make_subplots` automate. Knowing `subplots(rows, cols)` means you always have a way to lay out 2, 4, 6 or 9 charts coherently.

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 2)
assert axes.size == 4
print('ok')

## Challenge 8

Apply a log scale and verify it.

## Challenge 8 — Log scale and verify it

**The skill.** Data that spans orders of magnitude (prices, populations, responses) needs a logarithmic axis or the small values vanish. `ax.set_xscale('log')` switches the x-axis to a log scale, and `assert ax.get_xscale() == 'log'` reads the current setting back to confirm it — matplotlib stores axis scales as state you can query, and 'linear' is the default you must override.

**Think through.** With x = [1, 10, 100], a linear axis would crush the low values together; a log axis spaces them evenly. The same principle powered the log scales you saw on GDP scatter plots in earlier notebooks.

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
ax.plot([1, 10, 100], [1, 2, 3])
ax.set_xscale('log')
assert ax.get_xscale() == 'log'
print('ok')

## Challenge 9

Save a figure and check the file size is non-zero.

## Challenge 9 — Save a figure, prove it exists

**The skill.** Shipping a chart to a file is the endpoint of almost every analysis: `fig.savefig(path)` writes the figure to disk in a format inferred from the extension (`.png`). The challenge goes further and *verifies the write happened* — after creating the `artifacts` directory (`Path('artifacts'); out.mkdir(exist_ok=True)`) and saving `c.png`, it asserts `p.stat().st_size > 0`, i.e. the file is non-empty.

**Think through.** Asserting the output file exists and has bytes is a lightweight regression test — exactly how you would guard a pipeline that generates reports or model plots from crashing silently.

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
out = Path('artifacts'); out.mkdir(exist_ok=True)
fig, ax = plt.subplots()
ax.plot([1, 2], [1, 2])
p = out / 'c.png'
fig.savefig(p)
assert p.stat().st_size > 0
print('ok')

## Challenge 10

Choose an appropriate chart for five scenarios.

## Challenge 10 — Choose the right chart (the whole course, one dict)

**The skill.** The final challenge has no plotting at all — it is pure judgment, encoded as a `choice` dict mapping each scenario to the correct chart: one numeric variable → **histogram** (distribution), category comparison → **bar** (magnitude), two numerics → **scatter** (relationship), over time → **line** (trend), matrix of values → **heatmap** (two-way color grid). The assertion pins the time-series answer (`choice['over time'] == 'line'`) and the loop prints the full decision table.

**Think through.** This five-line dict is the summary of every session from Matplotlib to Plotly: the chart must serve the question. Memorize the mapping, and you instantly know which function to reach for in any real analysis.

In [ ]:
choice = {'one numeric variable': 'histogram',
          'category comparison': 'bar',
          'two numerics': 'scatter',
          'over time': 'line',
          'matrix of values': 'heatmap'}
assert choice['over time'] == 'line'
for k, v in choice.items():
    print(f'{k:<22} {v}')

## After the session

Time yourself: 10 minutes per challenge is a realistic screening pace. Re-do the ones that took longer.

## Summary

These ten challenges sharpened the day-to-day plotting habits that interviews and real projects punish you for missing. You practised explicit axes control (`subplots` + `set_title`), overlaying translucent histograms with legends, sorting categories before drawing bars, building correlation matrices for heatmaps, downsampling giant series before plotting, labelling with `bar_label`, constructing 2×2 facet grids, switching to log scales and verifying them, saving and validating output files — and finally the judgment call of choosing the right chart for the question. The promise of the notebook held: with the `assert` in every cell, each challenge teaches you to verify your work mechanically, not just eyeball it. Ten minutes per challenge is the pace to train for.